# 01 - Baseline model pre-training (raw experiments tanpa fine-tuning)

**IF4071 fase 2: ASR bahasa Sunda dengan wav2vec 2.0 dan CTC.**

## Description
*ASR pre-training* berarti memakai model baseline yang **sudah di-fine-tune untuk speech recognition**. Model ini terdiri dari dua bagian:
- **Encoder wav2vec 2.0**, yang di-train secara self-supervised pada raw audio. Encoder mengubah audio menjadi satu representasi suara untuk setiap potongan ~20 ms yang disebut *frame*.
- **CTC head** linear, yang pada setiap frame memberi skor untuk setiap karakter di vocab, ditambah satu token khusus bernama *blank* yang berarti "tidak ada karakter baru di frame ini".

Skor tersebut diubah menjadi teks dengan **greedy CTC decoding** dalam tiga langkah:
1. Pada setiap frame, pilih karakter dengan skor tertinggi.
2. Gabungkan karakter sama yang muncul berturut-turut menjadi satu.
3. Hapus semua *blank*.

Sebagai contoh: dengan `_` sebagai *blank*: hasil per frame `h h _ a a _ i i` menjadi `h _ a _ i` setelah langkah 2, lalu `hai` setelah langkah 3. *Blank* juga menjaga huruf kembar yang memang diucapkan, misalnya `l l _ l` menjadi `ll`, bukan `l`.

Karena CTC head dan tokenizer sudah tersedia di dalam model baseline, model dapat langsung mentranskripsi audio tanpa pelatihan tambahan.

**Tujuan:** menetapkan **WER/CER validasi** baseline kedua model sebelum fine-tuning.

| Model | Model baseline di Hugging Face |
|---|---|
| A | `cahya/wav2vec2-large-xlsr-sundanese` |
| B | `indonesian-nlp/wav2vec2-indonesian-javanese-sundanese` |

**Metode evaluasi:** pembagian data dilakukan secara speaker-independent (untuk menguji generalizability dan menghindari speaker bias) dan reproducible (seed 42, dibuat sekali dan disertakan pada dataset). Notebook ini hanya memakai speaker validation set dan test set tidak pernah dibaca (eksklusif untuk evaluasi akhir). CTC decoding secara greedy dan normalisasi teks identik untuk kedua model. Metrik utama memakai referensi ternormalisasi yang mempertahankan huruf e-acute. Metrik sekunder *tanpa aksen* (e-acute menjadi e) juga dilaporkan karena vocab asli Cahya tidak memuat e-acute sehingga model tersebut tidak mungkin menghasilkannya.

## Kaggle Environment

In [1]:
DATASET_ROOT = "/kaggle/input/datasets/haegenquinston/sundanese-wav2vec-asr/kaggle_export"
OUTPUT_ROOT = "/kaggle/working/asr_outputs"
SEED = 42

In [2]:
import importlib.util
import subprocess
import sys
from dataclasses import dataclass
from pathlib import Path
import json
import os
import platform
import random
import re
import sys
import unicodedata

missing = [p for p in ("jiwer", "accelerate", "soundfile") if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("Terpasang:", missing)

from huggingface_hub import model_info
from transformers import TrainingArguments
from transformers import Wav2Vec2CTCTokenizer, Wav2Vec2ForCTC, Wav2Vec2Processor
from transformers.trainer_utils import get_last_checkpoint
import hashlib
import jiwer
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import transformers

if not os.path.isdir(DATASET_ROOT):
    attached = os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else []
    raise FileNotFoundError(f"DATASET_ROOT {DATASET_ROOT!r} tidak ditemukan. Input terlampir: {attached}. "
                            "Isi dengan folder yang memuat splits/ dan audio/.")
os.makedirs(OUTPUT_ROOT, exist_ok=True)

Terpasang: ['jiwer']


## Helper Codes

In [3]:
NORMALIZATION_VERSION = "v1"


ALLOWED_CHARS = set("abcdefghijklmnopqrstuvwxyz\u00e9 ")


REMOVED_TAGS = {"ee", "ee..", "noise", "overlap", "s"}


REVIEW_TAGS = {"unclear"}


DIGITS = {"0": "nol", "1": "hiji", "2": "dua", "3": "tilu", "4": "opat",
          "5": "lima", "6": "genep", "7": "tujuh", "8": "dalapan", "9": "salapan"}


# Satu-satunya token gabungan huruf dan angka di korpus sehingga bentuk lisannya ditetapkan secara eksplisit.
SPOKEN_FORMS = {"s2": "es dua"}


_TAG_RE = re.compile(r"\[([^\]]*)\]")


_NUMBER_RE = re.compile(r"(?<![^\W\d_])\d+(?![^\W\d_])")


class NormalizationError(ValueError):
    pass


@dataclass
class NormalizedText:
    text: str
    needs_review: bool
    flags: tuple


def _spell_number(num: str) -> str:
    if len(num) > 2 or (len(num) > 1 and num.startswith("0")):
        raise NormalizationError(f"Angka {num!r} di luar pemetaan eksplisit 0-99.")
    n = int(num)
    if n < 10:
        return DIGITS[num]
    tens, unit = divmod(n, 10)
    if n == 10:
        return "sapuluh"
    if n == 11:
        return "sabelas"
    if tens == 1:
        return f"{DIGITS[str(unit)]} welas"
    return f"{DIGITS[str(tens)]} puluh" + (f" {DIGITS[str(unit)]}" if unit else "")


def normalize_transcript(raw: str) -> NormalizedText:
    if not isinstance(raw, str):
        raise NormalizationError(f"Transkripsi bukan string: {raw!r}")
    flags = []
    text = unicodedata.normalize("NFC", raw).lower()

    def handle_tag(m):
        inner = " ".join(m.group(1).split())
        if inner in REMOVED_TAGS:
            return " "
        if inner in REVIEW_TAGS:
            flags.append("unclear")
            return " "
        flags.append(f"unwrapped:{inner}")
        return f" {inner} "

    text = _TAG_RE.sub(handle_tag, text)
    for written, spoken in SPOKEN_FORMS.items():
        text = re.sub(rf"\b{re.escape(written)}\b", spoken, text)
    text = _NUMBER_RE.sub(lambda m: f" {_spell_number(m.group(0))} ", text)
    if re.search(r"\d", text):
        raise NormalizationError(f"Angka yang menempel pada huruf tidak didukung: {raw!r}")
    text = re.sub(r"[-\u2010-\u2015]", " ", text)
    text = "".join(ch if (ch.isalpha() or ch.isspace()) else " " for ch in text)
    text = " ".join(text.split())
    bad = sorted(set(text) - ALLOWED_CHARS)
    if bad:
        raise NormalizationError(f"Karakter tidak didukung {bad} setelah normalisasi: {raw!r}")
    return NormalizedText(text, "unclear" in flags, tuple(flags))


def normalize_text(raw: str) -> str:
    return normalize_transcript(raw).text


def fold_diacritics(text: str) -> str:
    """Hanya dipakai untuk metrik sekunder tanpa aksen dan tidak pernah dipakai sebagai target latih."""
    return text.replace("\u00e9", "e")


def load_audio(path, expected_sr: int = 16000) -> np.ndarray:
    """Memuat gelombang mono float32 tanpa resampling dan menolak format yang tidak sesuai."""
    wav, sr = sf.read(str(path), dtype="float32", always_2d=True)
    if sr != expected_sr:
        raise ValueError(f"{path}: laju sampel {sr} != {expected_sr}")
    if wav.shape[1] != 1:
        raise ValueError(f"{path}: diharapkan mono, ditemukan {wav.shape[1]} kanal")
    if wav.shape[0] == 0:
        raise ValueError(f"{path}: audio kosong")
    return wav[:, 0]


def pad_ctc_labels(label_ids: list, pad_value: int = -100) -> np.ndarray:
    """Posisi padding bernilai -100 agar diabaikan oleh loss CTC."""
    out = np.full((len(label_ids), max(len(x) for x in label_ids)), pad_value, dtype=np.int64)
    for i, ids in enumerate(label_ids):
        out[i, : len(ids)] = ids
    return out


def _clean(s) -> str:
    return " ".join(str(s).split())


def compute_error_rates(references, predictions, normalize: bool = True) -> dict:
    """Menghitung WER dan CER sebagai total edit dibagi total unit referensi. Prediksi kosong dihitung sebagai penghapusan penuh."""
    refs, hyps = list(references), [_clean(h) for h in predictions]
    if len(refs) != len(hyps):
        raise ValueError(f"{len(refs)} referensi vs {len(hyps)} prediksi")
    if not refs:
        raise ValueError("Tidak ada ujaran untuk dinilai")
    refs = [normalize_text(r) if normalize else _clean(r) for r in refs]
    if any(r == "" for r in refs):
        raise ValueError("Referensi kosong ditemukan; perbaiki manifes")
    wo = jiwer.process_words(refs, hyps)
    co = jiwer.process_characters(refs, hyps)
    return {
        "wer": wo.wer, "cer": co.cer, "n_utterances": len(refs),
        "word_ref_count": wo.hits + wo.substitutions + wo.deletions,
        "char_ref_count": co.hits + co.substitutions + co.deletions,
        "word_sub": wo.substitutions, "word_del": wo.deletions, "word_ins": wo.insertions,
        "char_sub": co.substitutions, "char_del": co.deletions, "char_ins": co.insertions,
    }


def verify_saved_split(split_dir) -> dict:
    """Memastikan CSV tersimpan konsisten dengan metadata dan tidak ada penutur maupun ujaran yang tumpang tindih."""
    d = Path(split_dir)
    meta = json.loads((d / "split_metadata.json").read_text())
    seen = {}
    for name in ("train", "val", "test"):
        if (d / f"{name}.csv").exists():
            df = pd.read_csv(d / f"{name}.csv")
            if set(df["speaker_id"]) != set(meta["subsets"][name]["speakers"]):
                raise AssertionError(f"Penutur {name}.csv berbeda dari split_metadata.json")
            seen[name] = df
    names = list(seen)
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            if not set(seen[a]["speaker_id"]).isdisjoint(seen[b]["speaker_id"]):
                raise AssertionError(f"Penutur tumpang tindih antara {a} dan {b}")
            if not set(seen[a]["file"]).isdisjoint(seen[b]["file"]):
                raise AssertionError(f"Ujaran tumpang tindih antara {a} dan {b}")
    return meta


def load_split(dataset_root, name: str, check_audio: bool = True) -> pd.DataFrame:
    """Memuat CSV pembagian dan menambahkan `abs_audio_path` relatif terhadap dataset_root."""
    root = Path(dataset_root)
    df = pd.read_csv(root / "splits" / f"{name}.csv")
    df["abs_audio_path"] = df["audio_path"].map(lambda p: str(root / p))
    if check_audio:
        missing = [p for p in df["abs_audio_path"] if not Path(p).exists()]
        if missing:
            raise FileNotFoundError(f"{len(missing)} berkas audio tidak ditemukan di {root}, mis. {missing[:3]}")
    return df


def env_info() -> dict:
    info = {"python": sys.version.split()[0], "platform": platform.platform(),
            "torch": torch.__version__, "transformers": transformers.__version__,
            "cuda_available": torch.cuda.is_available(), "cuda": torch.version.cuda}
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        info.update(gpu=p.name, gpu_mem_gb=round(p.total_memory / 1e9, 1), n_gpus=torch.cuda.device_count())
    return info


def gpu_mem() -> str:
    if not torch.cuda.is_available():
        return "tanpa CUDA"
    return f"terpakai {torch.cuda.memory_allocated() / 1e9:.2f} GB / dicadangkan {torch.cuda.memory_reserved() / 1e9:.2f} GB"


def load_model_and_processor(model_id: str, device: str = "cuda"):
    processor = Wav2Vec2Processor.from_pretrained(model_id)
    return Wav2Vec2ForCTC.from_pretrained(model_id).to(device), processor


def model_revision(model_id: str):
    try:
        return model_info(model_id).sha
    except Exception:  # luring, atau checkpoint dari dataset Kaggle lokal
        return None


def tokenizer_audit(processor, model, texts) -> dict:
    """Melaporkan vocab dan karakter target yang tidak dapat di-encode oleh tokenizer."""
    tok = processor.tokenizer
    vocab = tok.get_vocab()
    needed = set("".join(texts).replace(" ", ""))
    missing = sorted(c for c in needed if c not in vocab)
    return {
        "vocab_size": len(vocab), "model_vocab_size": model.config.vocab_size,
        "vocab": sorted(vocab, key=vocab.get), "word_delimiter": tok.word_delimiter_token,
        "pad_token_id": tok.pad_token_id, "unk_token": tok.unk_token,
        "expected_sampling_rate": processor.feature_extractor.sampling_rate,
        "do_normalize": processor.feature_extractor.do_normalize,
        "missing_target_chars": missing,
        "ctc_loss_reduction": model.config.ctc_loss_reduction,
    }


def add_missing_chars(processor, model, chars, tmp_dir):
    """Menambahkan karakter yang hilang sebagai token CTC baru dan memperbesar lm_head.

    Tokenizer dibangun ulang dari vocab yang diperluas agar token baru benar-benar menjadi
    bagian vocab dan ikut tersimpan. Baris lm_head asli dipertahankan dan hanya baris baru
    yang diinisialisasi.
    """
    old = processor.tokenizer
    vocab = dict(old.encoder)  # Memakai vocab dasar karena get_vocab juga memuat token tambahan bos dan eos
    chars = [c for c in chars if c not in old.get_vocab()]
    if not chars:
        return processor, model
    old_n = model.lm_head.out_features
    if old_n != len(vocab) or max(vocab.values()) != old_n - 1:
        raise RuntimeError(f"lm_head ({old_n}) dan vocab tokenizer ({len(vocab)}) tidak selaras; "
                           "perluasan tidak aman.")
    for i, c in enumerate(chars):
        vocab[c] = old_n + i
    Path(tmp_dir).mkdir(parents=True, exist_ok=True)
    vocab_file = Path(tmp_dir) / "vocab.json"
    vocab_file.write_text(json.dumps(vocab, ensure_ascii=False))
    tok = Wav2Vec2CTCTokenizer(str(vocab_file), unk_token=old.unk_token, pad_token=old.pad_token,
                               bos_token=old.bos_token, eos_token=old.eos_token,
                               word_delimiter_token=old.word_delimiter_token)
    processor = Wav2Vec2Processor(feature_extractor=processor.feature_extractor, tokenizer=tok)
    old_head = model.lm_head
    new_head = torch.nn.Linear(old_head.in_features, len(vocab))
    with torch.no_grad():
        new_head.weight[:old_n] = old_head.weight
        new_head.bias[:old_n] = old_head.bias
        torch.nn.init.normal_(new_head.weight[old_n:], std=0.02)
        torch.nn.init.zeros_(new_head.bias[old_n:])
    model.lm_head = new_head.to(old_head.weight.device)
    model.config.vocab_size = len(vocab)
    return processor, model


def count_unk_targets(processor, texts) -> int:
    """Menghitung target yang masih menjadi token UNK setelah encoding dan nilainya harus nol."""
    unk = processor.tokenizer.unk_token_id
    return sum(unk in processor.tokenizer(t).input_ids for t in texts)


def transcribe_files(model, processor, paths, batch_size: int = 4, device: str = "cuda", fp16: bool = True):
    """Dekode CTC greedy dengan berkas terpanjang lebih dulu agar padding minimal, lalu urutan asli dipulihkan."""
    sr = processor.feature_extractor.sampling_rate
    waves = [load_audio(p, sr) for p in paths]
    order = np.argsort([-len(w) for w in waves])
    preds = [None] * len(waves)
    model.eval()
    for i in range(0, len(order), batch_size):
        idx = order[i:i + batch_size]
        feats = processor.feature_extractor([waves[j] for j in idx], sampling_rate=sr,
                                            padding=True, return_tensors="pt", return_attention_mask=True)
        extra = {"attention_mask": feats["attention_mask"].to(device)} if "attention_mask" in feats else {}
        with torch.no_grad(), torch.autocast("cuda", enabled=fp16 and device == "cuda"):
            logits = model(feats["input_values"].to(device), **extra).logits
        for j, row in zip(idx, torch.argmax(logits, dim=-1).cpu()):
            preds[j] = processor.tokenizer.decode(row, skip_special_tokens=True)
    return [" ".join(p.split()) for p in preds]


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def require_cuda() -> None:
    if not torch.cuda.is_available():
        raise RuntimeError("GPU CUDA tidak terdeteksi. Di Kaggle: Settings > Accelerator > GPU T4. "
                           "Fine-tuning penuh di CPU sengaja tidak didukung.")


class CTCDataset:
    """Memuat gelombang 16 kHz saat dibutuhkan dan mengembalikan input ternormalisasi beserta id label."""

    def __init__(self, df: pd.DataFrame, processor):
        self.df, self.processor = df.reset_index(drop=True), processor
        self.sr = processor.feature_extractor.sampling_rate

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        wav = load_audio(row["abs_audio_path"], self.sr)
        return {"input_values": self.processor.feature_extractor(wav, sampling_rate=self.sr).input_values[0],
                "labels": self.processor.tokenizer(row["transcription_normalized"]).input_ids}


@dataclass
class CTCDataCollator:
    """Padding dinamis yang mengisi gelombang dengan nol dan label dengan nilai yang diabaikan CTC."""
    processor: object

    def __call__(self, features):
        batch = self.processor.feature_extractor.pad(
            [{"input_values": f["input_values"]} for f in features], padding=True, return_tensors="pt")
        batch["labels"] = torch.from_numpy(pad_ctc_labels([f["labels"] for f in features], -100))
        return batch


def make_compute_metrics(processor):
    def compute(pred):
        ids = np.argmax(pred.predictions, axis=-1)
        label_ids = np.where(pred.label_ids == -100, processor.tokenizer.pad_token_id, pred.label_ids)
        hyp = processor.batch_decode(ids)
        ref = processor.batch_decode(label_ids, group_tokens=False)
        r = compute_error_rates(ref, hyp, normalize=False)
        return {"wer": r["wer"], "cer": r["cer"]}
    return compute


def build_training_args(output_dir, cfg: dict, **overrides) -> TrainingArguments:
    t = cfg["training"]
    kw = dict(
        output_dir=str(output_dir), learning_rate=float(t["learning_rate"]),
        num_train_epochs=t["num_train_epochs"],
        per_device_train_batch_size=t["per_device_train_batch_size"],
        per_device_eval_batch_size=t["per_device_eval_batch_size"],
        gradient_accumulation_steps=t["gradient_accumulation_steps"],
        weight_decay=t["weight_decay"],
        fp16=t["fp16"], gradient_checkpointing=t["gradient_checkpointing"],
        save_strategy="epoch", save_total_limit=t["save_total_limit"], logging_steps=10,
        load_best_model_at_end=True, metric_for_best_model=t["metric_for_best_model"],
        greater_is_better=t["greater_is_better"], seed=cfg["seed"], data_seed=cfg["seed"],
        report_to="none", remove_unused_columns=False, dataloader_num_workers=2,
    )
    # Nama argumen warmup dan strategi evaluasi berbeda antar versi Transformers
    fields = TrainingArguments.__dataclass_fields__
    kw["warmup_ratio" if "warmup_ratio" in fields else "warmup_steps"] = t["warmup_ratio"]
    kw["eval_strategy" if "eval_strategy" in fields else "evaluation_strategy"] = "epoch"
    return TrainingArguments(**{**kw, **overrides})


def last_checkpoint(output_dir):
    d = Path(output_dir)
    return get_last_checkpoint(str(d)) if d.exists() else None


def history_frame(trainer) -> pd.DataFrame:
    return pd.DataFrame(trainer.state.log_history)


def plot_history(hist: pd.DataFrame):
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
    train = hist[hist["loss"].notna()]
    ev = hist[hist["eval_loss"].notna()]
    ax[0].plot(train["epoch"], train["loss"], label="loss latih")
    ax[0].plot(ev["epoch"], ev["eval_loss"], "o-", label="loss validasi")
    ax[1].plot(ev["epoch"], ev["eval_wer"], "o-", color="C1")
    ax[2].plot(ev["epoch"], ev["eval_cer"], "o-", color="C2")
    for a, title in zip(ax, ("Loss", "WER validasi", "CER validasi")):
        a.set_title(title)
        a.set_xlabel("epoch")
        a.grid(alpha=.3)
    ax[0].legend()
    fig.tight_layout()
    return fig


def save_artifacts(trainer, processor, out_dir, cfg_record: dict, val_df, preds, metrics: dict) -> None:
    out_dir = Path(out_dir)
    trainer.save_model(str(out_dir / "model"))
    processor.save_pretrained(str(out_dir / "processor"))
    (out_dir / "training_config.json").write_text(json.dumps(cfg_record, indent=2, ensure_ascii=False, default=str))
    history_frame(trainer).to_csv(out_dir / "training_history.csv", index=False)
    pd.DataFrame({"file": val_df["file"], "speaker_id": val_df["speaker_id"],
                  "reference": val_df["transcription_normalized"], "prediction": preds}
                 ).to_csv(out_dir / "validation_predictions.csv", index=False)
    (out_dir / "validation_metrics.json").write_text(json.dumps(metrics, indent=2))


CFG = {'seed': 42,
 'sampling_rate': 16000,
 'normalization_version': 'v1',
 'training': {'learning_rate': 1e-05,
              'num_train_epochs': 10,
              'per_device_train_batch_size': 1,
              'per_device_eval_batch_size': 1,
              'gradient_accumulation_steps': 8,
              'weight_decay': 0.01,
              'warmup_ratio': 0.1,
              'fp16': True,
              'gradient_checkpointing': True,
              'freeze_feature_encoder': True,
              'save_total_limit': 2,
              'metric_for_best_model': 'eval_cer',
              'greater_is_better': False}}

In [4]:
assert torch.cuda.is_available(), "Aktifkan GPU: Settings > Accelerator > GPU T4"
DEVICE = "cuda"
set_seed(SEED)
print(json.dumps(env_info(), indent=2))
print(torch.cuda.get_device_name(0), "|", gpu_mem())

{
  "python": "3.13.16",
  "platform": "Linux-6.18.55+-x86_64-with-glibc2.39",
  "torch": "2.11.0+cu130",
  "transformers": "5.20.2",
  "cuda_available": true,
  "cuda": "13.0",
  "gpu": "Tesla T4",
  "gpu_mem_gb": 15.6,
  "n_gpus": 2
}
Tesla T4 | terpakai 0.00 GB / dicadangkan 0.00 GB


## Dataset Loading
Dataset sudah melalui process splitting agar konsisten untuk semua eksperimen.

In [5]:
meta = verify_saved_split(os.path.join(DATASET_ROOT, "splits"))
val = load_split(DATASET_ROOT, "val")
assert set(val.speaker_id) == set(meta["subsets"]["val"]["speakers"])
print("fingerprint manifes:", meta["manifest_fingerprint"][:16], "| seed:", meta["seed"])
print(f"validasi: {len(val)} ujaran, {val.duration.sum() / 60:.1f} menit, penutur {sorted(val.speaker_id.unique())}")
print(val.quality.value_counts().to_dict(), val.native_status.value_counts().to_dict())
val.head(3)[["file", "speaker_id", "transcription_normalized"]]

fingerprint manifes: ed2b6fbf950495ac | seed: 42
validasi: 172 ujaran, 28.1 menit, penutur ['S05', 'S10']
{'Good': 172} {'Native': 172}


,file,speaker_id,transcription_normalized
0,S05_T01_001.wav,S05,cing caritakeun atuh ti mimiti hudang isuk nep...
1,S05_T01_002.wav,S05,ari abdi mah poéna lumayan padet euy isuk huda...
2,S05_T01_003.wav,S05,sanggeus kitu abdi mimiti gawé magang sok aya ...


## D. Pemuatan model
Setiap model dipakai bersama prosesor, tokenizer, dan kepala CTC aslinya. Audit tokenizer memeriksa apakah seluruh karakter target dapat di-encode.

In [6]:
MODELS = {
    "cahya": "cahya/wav2vec2-large-xlsr-sundanese",
    "indonesian_nlp": "indonesian-nlp/wav2vec2-indonesian-javanese-sundanese",
}
refs = val.transcription_normalized.tolist()
audits = {}
for key, model_id in MODELS.items():
    model, processor = load_model_and_processor(model_id, DEVICE)
    audit = tokenizer_audit(processor, model, refs)
    audits[key] = {k: v for k, v in audit.items() if k != "vocab"} | {"revision": model_revision(model_id)}
    print(f"{key}: vocab={audit['vocab_size']} sr={audit['expected_sampling_rate']} karakter target tak tercakup={audit['missing_target_chars']}")
    print("   vocab:", " ".join(audit["vocab"]))
    del model, processor
    torch.cuda.empty_cache()

preprocessor_config.json:   0%|          | 0.00/158 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.56k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/259 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.26GB            

... (13 outputs left out)

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.26GB            

model.safetensors: downloading bytes:           |  0.00B            

indonesian_nlp: vocab=30 sr=16000 karakter target tak tercakup=[]
   vocab: | a b c d e f g h i j k l m n o p q r s t u v w x y z é [UNK] [PAD]


## Baseline Inference
Greedy CTC decoding dengan fp16, batch kecil terurut panjang (audio berdurasi beragam ditangani dengan padding dan attention mask). Note: bila muncul `CUDA out of memory`, `BATCH_SIZE` perlu diturunkan menjadi 1.

In [7]:
BATCH_SIZE = 4
preds = {}
for key, model_id in MODELS.items():
    model, processor = load_model_and_processor(model_id, DEVICE)
    preds[key] = transcribe_files(model, processor, val.abs_audio_path.tolist(), batch_size=BATCH_SIZE, device=DEVICE)
    print(key, "selesai |", gpu_mem())
    del model, processor
    torch.cuda.empty_cache()

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

cahya selesai | terpakai 1.34 GB / dicadangkan 3.04 GB


Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

indonesian_nlp selesai | terpakai 1.27 GB / dicadangkan 2.99 GB


## Baseline Metrics
WER/CER tingkat korpus memakai `jiwer`. Kolom *tanpa aksen* mengubah e-acute menjadi e pada referensi dan prediksi sehingga penalti akibat ketiadaan token e-acute pada Cahya tidak ikut terhitung.

In [8]:
metrics = {}
for key in MODELS:
    strict = compute_error_rates(refs, preds[key], normalize=False)
    folded = compute_error_rates([fold_diacritics(r) for r in refs], [fold_diacritics(h) for h in preds[key]], normalize=False)
    metrics[key] = {"model_id": MODELS[key], "split": "val", "n_utterances": len(refs),
                    "wer": strict["wer"], "cer": strict["cer"],
                    "wer_e_folded": folded["wer"], "cer_e_folded": folded["cer"], "details": strict}
pd.DataFrame({k: {"WER": v["wer"], "CER": v["cer"], "WER (tanpa aksen)": v["wer_e_folded"], "CER (tanpa aksen)": v["cer_e_folded"]}
              for k, v in metrics.items()}).T.round(4)

,WER,CER,WER (tanpa aksen),CER (tanpa aksen)
cahya,0.6202,0.1662,0.5754,0.1469
indonesian_nlp,0.6812,0.1908,0.6442,0.1720


In [9]:
# Tiga ujaran pertama menurut urutan berkas agar contoh tidak dipilih secara subjektif.
for i in range(3):
    print("berkas:", val.file[i])
    print("  REF :", refs[i])
    for key in MODELS:
        print(f"  {key:15s}:", preds[key][i])

berkas: S05_T01_001.wav
  REF : cing caritakeun atuh ti mimiti hudang isuk nepi ka saré deui sapopoé téh sok ngapain waé
  cahya          : jeung caritake natuh timiwiti huda ngisuk nepi ka sare deui sapowe teh sok gapen wae
  indonesian_nlp : cing jaritaken atuh dime widy huda ngisuk nepi kasarya deuwi sapewe teh sok ngapen wae
berkas: S05_T01_002.wav
  REF : ari abdi mah poéna lumayan padet euy isuk hudang terus mandi jeung sarapan heula supaya aya tanaga biasana bari sarapan téh sok ngecek jadwal jeung pesen nu asup
  cahya          : aryabdi mah poe nalumen pahdeteih isu kudang turs mandi jeung sarapan helas payayatan aga gesana bari sarapan teh sok ngacekjadua jeung pese nu asup
  indonesian_nlp : aria abdima poena lu main padhet eh isuk kudang trus manijeng sarapan hela supaya ya tanagak ba sana bari sarapan teh sok ngecek jadwal jeung pesenu asup
berkas: S05_T01_003.wav
  REF : sanggeus kitu abdi mimiti gawé magang sok aya waé nu kudu dikerjakeun unggal poé jadi kudu bisa ngabag

## Result Saving

In [10]:
out = os.path.join(OUTPUT_ROOT, "baseline")
os.makedirs(out, exist_ok=True)
for key in MODELS:
    pd.DataFrame({"file": val.file, "speaker_id": val.speaker_id, "quality": val.quality,
                  "reference": refs, "prediction": preds[key]}).to_csv(f"{out}/{key}_validation_predictions.csv", index=False)
record = {"seed": SEED,
          "split_metadata": {"manifest_fingerprint": meta["manifest_fingerprint"], "val_speakers": meta["subsets"]["val"]["speakers"]},
          "normalization_version": meta["normalization_version"], "decoding": "greedy CTC",
          "env": env_info(), "models": metrics, "tokenizer_audit": audits}
with open(f"{out}/baseline_validation_metrics.json", "w") as f:
    json.dump(record, f, indent=2, ensure_ascii=False)
print(sorted(os.listdir(out)))

['baseline_validation_metrics.json', 'cahya_validation_predictions.csv', 'indonesian_nlp_validation_predictions.csv']


## Interpretation
Sel berikut membaca angka yang dihitung di atas (tidak ada nilai yang ditulis manual).

In [11]:
best_wer = min(metrics, key=lambda k: metrics[k]["wer"])
best_cer = min(metrics, key=lambda k: metrics[k]["cer"])
print(f"WER validasi terendah: {best_wer} ({metrics[best_wer]['wer']:.3f})")
print(f"CER validasi terendah: {best_cer} ({metrics[best_cer]['cer']:.3f})")
print("WER dan CER sepakat." if best_wer == best_cer else "WER dan CER TIDAK sepakat mengenai model yang lebih baik.")
for key, audit in audits.items():
    if audit["missing_target_chars"]:
        print(f"Catatan: tokenizer {key} tidak memuat {audit['missing_target_chars']} sehingga WER/CER ketatnya naik; lihat kolom tanpa aksen.")

WER validasi terendah: cahya (0.620)
CER validasi terendah: cahya (0.166)
WER dan CER sepakat.
Catatan: tokenizer cahya tidak memuat ['é'] sehingga WER/CER ketatnya naik; lihat kolom tanpa aksen.


Note: Angka ini adalah hasil zero-shot dari model yang belum pernah melihat penutur, topik, maupun fenomena ujaran spontan pada korpus kami (kata pengisi, alih kode). Baseline yang lebih baik tidak menjamin model yang lebih baik setelah fine-tuning: kedua model baseline di-fine-tune secara independen dan identik pada notebook 02 dan 03, dan tidak ada pemilihan model di sini.